# C04 技术指标与分散化
这一节把“回测框架”接到更具体的策略上。

我们不会同时讲很多指标，而是保留两个最典型的例子：
1. Bollinger Band：主案例
2. RSI：对照案例

最后再回答一个很实际的问题：
> 单个策略可能波动很大，能不能通过多资产分散化让净值更稳？


In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"
UNIVERSE = ["510300.XSHG", "510500.XSHG", "159915.XSHE", "518880.XSHG"]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import rqdatac

# 方式1（推荐课堂演示）：把教育版 license 直接粘贴到 PASSWD
PASSWD = "在这里粘贴你的教育版license"

# 方式2（不想明文写在 notebook）：注释掉上面一行，改用环境变量
# import os
# PASSWD = os.getenv("RQDATAC_LICENSE", "")

if PASSWD and ("在这里粘贴" not in PASSWD):
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


## 1) 为什么技术指标适合放在这里讲
到这一节时，学生已经见过：
- `rolling()`
- `shift()`
- 波动率
- 均线

所以这时引入技术指标非常自然，因为很多技术指标本质上就是：
- 对价格做滚动均值
- 对波动做滚动标准差
- 对上涨/下跌幅度做再加工


In [ ]:
# 这里继续使用几只宽基ETF做演示，便于讨论“多资产”和“分散化”
assets = ["510300.XSHG", "510500.XSHG", "159915.XSHE", "513100.XSHG"]
price = rqdatac.get_price(
    assets,
    start_date=START_DATE,
    end_date=END_DATE,
    fields=["close"],
    adjust_type="pre",
    expect_df=True,
).reset_index()

price = price.sort_values(["order_book_id", "date"]).copy()
price.head()


## 2) 布林线：主案例
布林线的直觉非常适合课堂讲解：
- 用均线描述价格的“中枢”
- 用标准差描述“正常波动范围”
- 当价格偏离太远时，可能意味着超涨或超跌

这也是为什么它很适合用来讲均值回复。


In [ ]:
# 先单独看 510300 上的布林线长什么样
boll_demo = price.loc[price["order_book_id"] == "510300.XSHG"].copy()
boll_demo["mid"] = boll_demo["close"].rolling(20).mean()
boll_demo["std"] = boll_demo["close"].rolling(20).std()
boll_demo["upper"] = boll_demo["mid"] + 2 * boll_demo["std"]
boll_demo["lower"] = boll_demo["mid"] - 2 * boll_demo["std"]

fig, ax = plt.subplots(figsize=(10, 4))
boll_demo.plot(x="date", y="close", ax=ax, label="close", alpha=0.8)
boll_demo.plot(x="date", y="mid", ax=ax, label="mid")
boll_demo.plot(x="date", y="upper", ax=ax, label="upper")
boll_demo.plot(x="date", y="lower", ax=ax, label="lower")
ax.set_title("510300 布林线示意图")
plt.show()


### 2.1 这张图想让学生看到什么
这里不是要背“上轨中轨下轨”的定义，而是让大家看见：
1. 均线本质上就是一个平滑后的中枢
2. 上下轨是在中枢周围加上一层“正常波动范围”
3. 所谓信号，往往就是根据价格和这些参考线之间的关系来生成的


In [ ]:
# 定义一个最小版本的布林线仓位信号：
# 跌破下轨后做多，回到中轨附近就平仓
def boll_signal(series, n=20, width=2.0):
    mid = series.rolling(n).mean()
    std = series.rolling(n).std()
    upper = mid + width * std
    lower = mid - width * std

    signal = pd.Series(0.0, index=series.index)
    signal[series < lower] = 1.0
    signal = signal.replace(0, np.nan).ffill().fillna(0)
    signal[series >= mid] = 0.0
    return signal


In [ ]:
# 在多资产上运行布林线策略，并做等权分散
close_wide = price.pivot(index="date", columns="order_book_id", values="close")
ret_wide = close_wide / close_wide.shift(1) - 1

boll_pos = pd.DataFrame(index=close_wide.index, columns=close_wide.columns, dtype=float)
for col in close_wide.columns:
    boll_pos[col] = boll_signal(close_wide[col])

boll_ret = (boll_pos.shift(1) * ret_wide).mean(axis=1).fillna(0)
boll_nav = (1 + boll_ret).cumprod()
boll_nav.tail()


## 3) RSI：对照案例
RSI 的语言和布林线不完全一样，它更关注：
- 最近一段时间上涨力度强不强
- 下跌力度强不强

所以它很适合在这一节里作为“第二种技术指标思路”的对照。


In [ ]:
def rsi(series, n=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    avg_up = up.rolling(n).mean()
    avg_down = down.rolling(n).mean()
    rs = avg_up / avg_down
    return 100 - 100 / (1 + rs)

rsi_value = pd.DataFrame(index=close_wide.index, columns=close_wide.columns, dtype=float)
rsi_pos = pd.DataFrame(index=close_wide.index, columns=close_wide.columns, dtype=float)

for col in close_wide.columns:
    rsi_value[col] = rsi(close_wide[col])
    sig = pd.Series(0.0, index=close_wide.index)
    sig[rsi_value[col] < 30] = 1.0
    sig = sig.replace(0, np.nan).ffill().fillna(0)
    sig[rsi_value[col] > 50] = 0.0
    rsi_pos[col] = sig

rsi_ret = (rsi_pos.shift(1) * ret_wide).mean(axis=1).fillna(0)
rsi_nav = (1 + rsi_ret).cumprod()
rsi_nav.tail()


In [ ]:
# 把两条策略曲线放在一起看
perf = pd.DataFrame({
    "Bollinger": boll_nav,
    "RSI": rsi_nav,
})

fig, ax = plt.subplots(figsize=(10, 4))
perf.plot(ax=ax)
ax.set_title("技术指标策略净值对比")
plt.show()


## 3.5 再补一个旧讲义里常见的指标：MACD
这一节不一定要把 MACD 也完整做成策略，但它值得作为“第三种指标语言”补充一下。

原因是：
- Bollinger 更像价格偏离
- RSI 更像强弱对比
- MACD 更像均线差值的趋势变化


In [ ]:
def macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    hist = dif - dea
    return dif, dea, hist

macd_demo = close_wide["510300.XSHG"].dropna()
dif, dea, hist = macd(macd_demo)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
macd_demo.plot(ax=axes[0], title="510300 Close")
dif.plot(ax=axes[1], label="DIF")
dea.plot(ax=axes[1], label="DEA")
hist.plot(ax=axes[1], kind="bar", alpha=0.3, label="HIST")
axes[1].legend()
plt.tight_layout()
plt.show()


### 3.6 为什么这里只演示 MACD，不完整回测
因为这一节的重点已经够多了：
- 布林线
- RSI
- 分散化

把 MACD 作为补充演示，能让学生知道：
- 技术指标远不止两种
- 但所有指标都要最终落回“仓位信号”和“组合收益”


## 4.2 技术指标和多因子其实是连着的
从研究流程看，技术指标策略和后面的多因子并不是两门完全分开的课。

技术指标也可以被当成因子：
- RSI 可以变成一个横截面特征
- MACD 也可以变成一个动量/趋势信号

这也是为什么下一节从单因子到多因子会接得很自然。


In [ ]:
# 把 RSI 当成“因子”看一眼截面形态
rsi_snapshot = rsi_value.loc[rsi_value.index.intersection(rebalance_dates)[-1]].dropna().sort_values()
rsi_snapshot.head(), rsi_snapshot.tail()


### 3.1 为什么要保留一个“对照策略”
因为教学上很重要的一点是：
- 不是为了证明某个指标永远最好
- 而是让学生看到，不同指标本质上是在用不同方式概括价格信息

同样的数据，换一种信号语言，结果就可能不一样。


## 4) 分散化在这里体现在哪里
这一节最想讲明白的，不只是“做了两个指标”，而是：
- 我们不是只在一只资产上做策略
- 而是在多只资产上同时跑，再做等权聚合

这就是最朴素的分散化思想。


In [ ]:
# 对比“单只资产策略”和“多资产等权策略”
single_boll = (boll_pos["510300.XSHG"].shift(1) * ret_wide["510300.XSHG"]).fillna(0)
single_boll_nav = (1 + single_boll).cumprod()

compare_nav = pd.DataFrame({
    "Single_510300": single_boll_nav,
    "Diversified_Boll": boll_nav,
})

fig, ax = plt.subplots(figsize=(10, 4))
compare_nav.plot(ax=ax)
ax.set_title("单资产 vs 分散化")
plt.show()


### 4.1 怎么解释这张分散化对比图
如果多资产曲线更平滑，通常说明：
- 不同资产的交易节奏不完全同步
- 某一只资产失效时，其他资产可能还在工作

这不是说分散化一定更赚钱，而是说它通常更接近真实投资组合的思维。


## 5) 课上小练习


### 练习 1：自己改一下 Bollinger 参数
要求：
1. 把 `n=20` 改成 `n=10`
2. 重新计算布林线信号和净值
3. 观察曲线会不会变得更敏感


In [ ]:
# 练习 1：学生现场自己写


In [ ]:
# 参考答案
boll_pos_10 = pd.DataFrame(index=close_wide.index, columns=close_wide.columns, dtype=float)
for col in close_wide.columns:
    boll_pos_10[col] = boll_signal(close_wide[col], n=10)

boll_ret_10 = (boll_pos_10.shift(1) * ret_wide).mean(axis=1).fillna(0)
boll_nav_10 = (1 + boll_ret_10).cumprod()
boll_nav_10.tail()


### 练习 2：比较是否分散化
要求：
1. 保留 `single_boll_nav` 和 `boll_nav`
2. 把它们画在同一张图上
3. 说一说哪条曲线更平滑，为什么


In [ ]:
# 练习 2：学生现场自己写


In [ ]:
# 参考答案
compare_ex2 = pd.DataFrame({
    "single": single_boll_nav,
    "diversified": boll_nav,
})
compare_ex2.plot(figsize=(10, 4), title="是否分散化的净值对比")
plt.show()


## 小结
这一节的关键词是：**指标策略 + 分散化**。

本节最重要的不是记住某个指标公式，而是理解三件事：
1. 技术指标本质上还是对价格序列做加工
2. 指标要通过仓位信号才能进入回测
3. 多资产组合往往比单一资产更接近真实可投资框架

本节常见易错点：
1. 只会画指标，不会把指标变成仓位信号
2. 忽略 `shift(1)`，把未来信息直接用了
3. 只看单一资产，不考虑分散化